![Logo SENATI](logo_senati.png)

**SENATI | Escuela de Tecnologías de la Información**  
**Taller: Machine Learning y Deep Learning**  
*Instructor: Ing. Gunar Castro Iñigo*

---

**Trabajo desarrollado por:** COCA CCORPA LORENA ABIGAIL  
**Institución:** SENATI · 2026  
**Actividad:** Sistema de estimación de pedidos de kits de robótica  
**Nota:** Los datos suministrados por el taller son simulados; los resultados y pruebas son cálculos realizados sobre esos CSV.

# Sistema de estimación de pedidos de kits de robótica — caso práctico intermedio

**Contexto:** Una tienda educativa quiere anticipar pedidos semanales. El número de consultas y la inversión publicitaria pueden asociarse con más ventas; el indicador de campaña escolar toma 0 o 1.

**Objetivo:** aplicar regresión múltiple desde lectura de datos hasta una web pequeña. Aquí no se entrega el resultado completo: debes demostrar lo que aprendiste durante los cuatro proyectos.

## Paso 1. Explorar el CSV
Busca las columnas de entrada `X`: consultas_web, inversion_publicidad_soles, campana_escolar, y la respuesta `y`: `pedidos_kits`.

**¿Qué podemos preguntar al grupo?** ¿Qué significa cada variable en la vida real? ¿Cuál sería el peligro de usar esta predicción sin verificarla?

### Respuestas cortas

- `consultas_web`: consultas por internet; `inversion_publicidad_soles`: gasto en publicidad; `campana_escolar`: 0 si no hay campaña y 1 si la hay. La respuesta son los pedidos estimados.
- **¿Cuál es el peligro de no verificarla?** Tomar decisiones basándose en un cálculo equivocado, especialmente porque los datos del curso son simulados.

<div style="background-color:#FFF0A8;color:#463600;border-left:7px solid #D6A400;padding:14px 18px;border-radius:9px;margin:10px 0 16px"><strong>🟨 INICIO DEL DESARROLLO DE LA ACTIVIDAD</strong><br>Desde aquí se realiza la lectura del CSV y el entrenamiento del modelo. Se conserva el enunciado original y el código se ejecuta en orden.</div>

In [1]:
import pandas as pd

datos = pd.read_csv('datos.csv')
print(datos.head())
print(datos.isna().sum())

  semana  consultas_web  ...  campana_escolar  pedidos_kits
0   S001             31  ...                1            22
1   S002             51  ...                0            20
2   S003             27  ...                0            17
3   S004             45  ...                0            21
4   S005             40  ...                1            16

[5 rows x 5 columns]
semana                        0
consultas_web                 0
inversion_publicidad_soles    0
campana_escolar               0
pedidos_kits                  0
dtype: int64


## Paso 2. Entrenar (completar)
Adapta el procedimiento de las prácticas 2 y 4. No copies resultados de otros compañeros: explica qué variables usaste, cuántas filas reservaste y cómo interpretas el MAE.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

columnas = ['consultas_web', 'inversion_publicidad_soles', 'campana_escolar']
objetivo = 'pedidos_kits'
X = datos[columnas]
y = datos[objetivo]

# Entrenamiento y evaluación (80/20, random_state=42, sin fuga de datos)
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)
modelo = LinearRegression()
modelo.fit(X_ent, y_ent)
estimaciones = modelo.predict(X_prueba)
mae = mean_absolute_error(y_prueba, estimaciones)
r2 = r2_score(y_prueba, estimaciones)
print('Registros:', len(datos),'| Entrenamiento:', len(X_ent),'| Prueba:',len(X_prueba))
print('MAE:', round(mae,3),'| R²:',round(r2,4))
print('Intercepto:',round(modelo.intercept_,4))
for nombre, peso in zip(columnas,modelo.coef_):
    print(f'{nombre}: {peso:.5f}')
print('MAE de referencia (media de entrenamiento):',round(mean_absolute_error(y_prueba, [y_ent.mean()]*len(y_prueba)),3))

Registros: 110 | Entrenamiento: 88 | Prueba: 22
MAE: 2.954 | R²: 0.7911
Intercepto: 1.5385
consultas_web: 0.34630
inversion_publicidad_soles: 0.00670
campana_escolar: 5.01821
MAE de referencia (media de entrenamiento): 6.661


## Paso 3. Aplicación web
Tu página necesita título realista, 3 campos con unidades, botón “Calcular”, resultado y una advertencia de estimación. Usa Flask, HTML y CSS (sin JavaScript). Separa los archivos en las carpetas habituales.

**Aplicación preparada:** `app.py`, `templates/index.html` y `static/estilos.css`. El formulario usa los tres datos de entrada, muestra una estimación y aclara que el cálculo es educativo. La comprobación del sitio en navegador queda por registrar como evidencia.

## Paso 4. Pruebas obligatorias
- Semana con 20 consultas, S/ 200 y sin campaña
- Semana con 55 consultas, S/ 800 y con campaña

¿Qué valores devolvió tu sistema? ¿Tienen sentido? ¿En qué circunstancias la predicción podría fallar?

<div style="background-color:#FFF0A8;color:#463600;border-left:7px solid #D6A400;padding:14px 18px;border-radius:9px;margin:10px 0 16px"><strong>🟨 INICIO DEL DESARROLLO DE LA ACTIVIDAD</strong><br>Desde aquí se realiza las dos pruebas solicitadas. Se conserva el enunciado original y el código se ejecuta en orden.</div>

## Resultados, pruebas obligatorias y conclusiones

In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
pruebas = pd.DataFrame([(20, 200, 0), (55, 800, 1)],columns=columnas)
pruebas['prediccion'] = modelo.predict(pruebas[columnas])
print(pruebas.to_string(index=False))
resultado_prueba = X_prueba.copy()
resultado_prueba['real'] = y_prueba.values
resultado_prueba['estimado'] = estimaciones
resultado_prueba.to_csv('predicciones_prueba.csv',index=False)
with open('modelo.json','w',encoding='utf-8') as archivo:
    json.dump({'intercepto':float(modelo.intercept_),'coeficientes':[float(v) for v in modelo.coef_], 'columnas':columnas,'objetivo':objetivo},archivo,indent=2,ensure_ascii=False)
print('Guardados modelo.json y predicciones_prueba.csv')

 consultas_web  inversion_publicidad_soles  campana_escolar  prediccion
            20                         200                0    9.804362
            55                         800                1   30.962658
Guardados modelo.json y predicciones_prueba.csv


### Respuestas cortas

**Prueba 1:** 20 consultas, S/ 200, sin campaña → **9.80 pedidos**.  
**Prueba 2:** 55 consultas, S/ 800, con campaña → **30.96 pedidos**.  
**¿Tienen sentido?** Sí, más consultas y publicidad se asocian con más pedidos en este ejemplo.  
**¿Cuándo puede fallar?** Cuando cambian las condiciones de demanda o los datos no reflejan la realidad.

## Evidencias y evaluación
Entregar notebook ejecutado, CSV, script de entrenamiento, `modelo.json`, aplicación Flask, dos capturas de prueba y conclusiones propias. Criterios: datos 3 pts; modelo 5 pts; métricas 4 pts; aplicación 5 pts; documentación 3 pts.

**Interpretación:** El MAE es el error absoluto medio en las unidades de la variable objetivo; R² mide el ajuste frente a la referencia de la media. Los coeficientes describen asociaciones dentro de los datos simulados, no causalidad. **Limitaciones:** datos sintéticos, posibles cambios en costos y demanda, variables omitidas; la validez requiere datos reales y evaluación externa. Las dos pruebas pertenecen al enunciado original.